In [1]:
%load_ext autoreload
%autoreload 2

```{toctree}
:maxdepth: 1
:caption: Contents:

# Walkthrough Example - Data Fetcher

First, different python libraries are imported to handle the data retrieval and processing.

In [2]:
import pandas as pd
from datetime import datetime, timedelta
import xarray as xr

The different modules in the `oceanicospy.retrievals` subpackage are imported to showcase the data retrieval capabilities.

In [3]:
from oceanicospy.retrievals import UHSLCDownloader,CMDSDownloader,ERA5Downloader

# University of Hawaii Sea Level Center (UHSLC) Data Downloader

This code is intended to download hourly sea-level data from the University of Hawaii Sea Level Center (UHSLC). The `UHSLCDownloader` class is defined to handle the downloading process. Three parameters are required: the `station_id` for the station of interest and the `output_path` where the downloaded data will be saved, and the `output_filename` for the resulting CSV file. The station id can be found on the UHSLC website (https://uhslc.soest.hawaii.edu/stations/).

A ``difference_from_UTC`` parameter is also included to convert the raw UHSLC timestamps to local time in the cleaned DataFrame. For example, UTC-5 should be passed as ``-5``.

An start and end date can also be provided to trim the data after downloading. The dates should be passed as strings in a format that can be parsed by `pandas.to_datetime` (e.g. "2010-01-01"). If either of the dates is `None`, the series will not be trimmed at that end. 

In [ ]:
UHSLCDownloader_obj = UHSLCDownloader(station_id='007', output_path='../data/temp/uhslc', output_filename='station_007.csv')

In [9]:
UHSLCDownloader_obj = UHSLCDownloader(
    station_id='007',
    output_path='../data/temp/uhslc',
    output_filename='station_007.csv',
    start_datetime_local='2010-01-01',
    end_datetime_local='2020-12-31',
    difference_from_UTC=-5,
)

In [5]:
filepath = UHSLCDownloader_obj.download()

Downloaded station_007.csv to ..\data\temp\uhslc\station_007.csv.


The method ``UHSLCDownloader.download()`` returns ``filepath``, which is the path to the downloaded file. The file is saved in CSV format and can be read using pandas or any other data analysis tool. A short method is also included to quickly read the downloaded data into a pandas DataFrame ``UHSLCDownloader.clean_data``


In [6]:
df_clean_local = UHSLCDownloader_obj.clean_data(filepath)

In [7]:
df_clean_local.head()

,depth[m]
1969-05-18 10:00:00,1.563
1969-05-18 11:00:00,1.399
1969-05-18 12:00:00,1.292
1969-05-18 13:00:00,1.277
1969-05-18 14:00:00,1.417


# Copernicus Marine Data Store (CMDS) downloader

There is a class called `CMDSDownloader` that is designed to download data from the Copernicus Marine Data Store (CMDS). This class uses the `copernicusmarine` toolbox to submit spatial/temporal subset requests to the CMDS and download the resulting data. The class is initialized with the following parameters:
- `dataset_id`: The ID of the dataset to download (e.g., "dataset-id").
- `variables`: A list of variables to download (e.g., [``"variable1"``, ``"variable2"``]).
- spatial parameters for the region of interest (e.g., `lon_min`, `lon_max`, `lat_min`, `lat_max`).
- temporal parameters for the time range of interest (e.g., `start_date_local`, `end_date_local`).
- `difference_to_utc`: The time difference to UTC in hours (e.g., -5 for EST).
- `output_path`: The directory where the downloaded data will be saved.
- `file_format`: The format of the output file (e.g., "netcdf", "zarr").
- `filename`: The name of the output file (e.g., "output.nc").


## For winds
There is a classmethod that tailors the CMDSDownloader to download wind data from the CMDS. This method is called ``CMDSDownloader.for_winds`` and it takes the same parameters as the main class, but it automatically sets the `dataset_id` to the appropriate dataset for wind data and selects the relevant variables for wind analysis. This method simplifies the process of downloading wind data from the CMDS by pre-configuring the necessary parameters for this specific type of data.

```{note}

For wind the following product and variables are defined as default:    
- Product: ``cmems_obs-wind_glo_phy_nrt_l4_0.125deg_PT1H``
- Variables: [``"eastward_wind"``, ``"northward_wind"``]

In [8]:
winds_CMDS = CMDSDownloader.for_winds(
    lon_min=-82,
    lon_max=-81,
    lat_min=12,
    lat_max=13,
    start_datetime_local=datetime(2023, 12, 1, 0, 0),             
    end_datetime_local=datetime(2023, 12, 31, 23, 0),               
    difference_to_UTC=-5,                                       
    output_path="../data/temp/CMDS",
    output_filename="winds_CMDS.nc",                             
    file_format="netcdf",
)

In [ ]:
winds_CMDS_filepath = winds_CMDS.download()

With the method ``format_to_localtime``, the timestamps in the downloaded ERA5 dataset can be converted from UTC to local time. This is done by applying the specified time difference to UTC (e.g., -5 for EST) to the timestamps in the dataset. This method allows users to work with the data in their local time zone, which can be more convenient for analysis and interpretation.

In [ ]:
winds_CMDS.format_to_localtime()

In [ ]:
winds_CMDS_ds = xr.open_dataset(winds_CMDS_filepath) 
winds_CMDS_ds

## For waves

Similar to winds, there is a default configuration for downloading wave data from the CMDS. The classmethod ``CMDSDownloader.for_waves`` is designed to facilitate the download of wave data by pre-setting the appropriate `dataset_id` and selecting the relevant variables for wave analysis. This method allows users to easily access wave data from the CMDS without needing to manually configure the parameters.

```{note}

For wind the following product and variables are defined as default:    
- Product: ``cmems_obs-wave_glo_wav_anfc_0.083deg_PT3H``
- Variables: [``"VHM0"``, ``"VMDR"``, ``"VTPK"``]

In [ ]:
waves_CMDS = CMDSDownloader.for_waves(
    lon_min=-82,
    lon_max=-80,
    lat_min=12,
    lat_max=14,
    start_datetime_local=datetime(2024, 12, 1, 0, 0),             
    end_datetime_local=datetime(2024, 12, 31, 23, 0),               
    difference_to_UTC=-5,                                       
    output_path="../data/temp/CMDS",
    output_filename="waves_CMDS.nc",                             
    file_format="netcdf",
)

In [ ]:
waves_CMDS_filepath = waves_CMDS.download()

In [ ]:
waves_CMDS.format_to_localtime()

In [ ]:
waves_CMDS_ds = xr.open_dataset(waves_CMDS_filepath) 
waves_CMDS_ds

# ERA5 Downloader

ERA5 is the fifth generation ECMWF reanalysis for the global climate and weather. This product is provided by the European Centre for Medium-Range Weather Forecasts (ECMWF) and is available through the Copernicus Climate Data Store (CDS). The `ERA5Downloader` class is designed to facilitate the download of ERA5 data from the CDS. This class uses the `cdsapi` library to submit requests to the CDS and download the resulting data.

In [ ]:
variables = ["10m_u_component_of_wind","10m_v_component_of_wind",]

# Spatial domain (WGS84); San Andrés example
lon_min, lon_max = -81.90, -81.50   # West/East
lat_min, lat_max =  12.35,  12.75   # South/North

start_local = datetime(2023, 8, 1, 0, 0)
end_local   = datetime(2023, 8, 31, 23, 0)
utc_offset_hours = -5  # Local = UTC-5

# Output file
output_path = "../data/temp/ERA5/"

winds_ERA5 = ERA5Downloader(
    variables=variables,
    lon_min=lon_min,
    lon_max=lon_max,
    lat_min=lat_min,
    lat_max=lat_max,
    start_datetime_local=start_local,
    end_datetime_local=end_local,
    difference_to_UTC=utc_offset_hours,
    output_path=output_path,
    output_filename="winds_ERA5.nc",
    cdsapi_rc="/Users/franklinayala/.cdsapirc",
)

In [ ]:
winds_ERA5_filepath = winds_ERA5.download()

Similar to the CMDSDownloader, the method ``format_to_localtime`` is included to convert the timestamps in the downloaded ERA5 dataset from UTC to local time. This allows users to work with the data in their local time zone, which can be more convenient for analysis and interpretation.

In [ ]:
winds_ERA5.format_to_localtime()

The dataset is open with xarray and the dates should be in local time.

In [ ]:
winds_ERA5_ds = xr.open_dataset(winds_ERA5_filepath)
winds_ERA5_ds